# Hudi vector index — structure & inspection

This notebook helps you **inspect how a vector (IVF-style) index is represented** in Hudi: metadata table (MDT) partitions, Avro payload fields, and the on-disk layout under `gs://` or local paths.

**Prerequisites**

- **Spark** (local or Spark Connect) with Hudi + GCS if you use `gs://`
- Optional: `gsutil` or Google ADC for listing / `cat` on GCS without Spark Hadoop FS
- Optional: `numpy` for centroid byte decoding

Set paths via environment variables or edit the config cell below.

## 1. Logical index structure (IVF in MDT)

Hudi’s vector index metadata is modeled as **IVF-style** records in the **metadata table** (when that partition is enabled and populated):

| Concept | Role |
|--------|------|
| **Index partition** | One MDT partition per logical index, path prefix `vector_index_<name>` (see `HoodieTableMetadataUtil.PARTITION_NAME_VECTOR_INDEX_PREFIX`). |
| **Centroid table** | A single special record whose key is **`__centroids__`**. `centroidBytes` holds **K × D × 4** bytes (**float32 little-endian**), row-major cluster centroids. |
| **Assignments** | Other records map **record key → cluster id** (`clusterId` ≥ 0). Tombstones use `isDeleted`. |

Avro shape (see `HoodieMetadata.avsc` → `HoodieVectorIndexInfo`):

- `clusterId` (int): cluster for this row key; **-1** for the centroid-dump record.
- `centroidBytes` (bytes | null): non-null **only** for key `__centroids__`.
- `isDeleted` (bool): tombstone for assignments.

Your **data table** still stores the **VECTOR(dim)** column on records (Parquet + `hudi_type` metadata); the MDT index is a **separate** acceleration structure.

## 2. Typical on-disk layout

```text
<table_base>/
  .hoodie/
    hoodie.properties          # lists enabled MDT partitions, table props
    timeline/...
  <data partition dirs>/       # contains columnar files with VECTOR column
  ...

# If metadata table is enabled:
<table_base>/.hoodie/metadata/
  .hoodie/
    hoodie.properties
    timeline/...
  vector_index_<idxName>/      # vector index MDT partition (when built)
    <file groups>.parquet / log ...
```

Use the cells below to **list** paths and **print** `hoodie.properties` for your table.

### Diagram (IVF records in MDT)

```mermaid
flowchart LR
  subgraph DATA["Data table"]
    V["embedding column\n(Parquet + hudi_type=VECTOR(dim))"]
  end
  subgraph MDT["MDT partition vector_index_*"]
    CENT["key = __centroids__\nclusterId = -1\ncentroidBytes = K×D×4 float32 LE"]
    ASG["key = record key\nclusterId = 0..K-1\nisDeleted optional"]
  end
  V -.->|indexed by| ASG
  CENT --- ASG
```

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path
from typing import List, Optional

# --- Edit or export env vars ---
# GCS base used in your tests:
TABLE_BASE = os.environ.get("HOODIE_GCS_VECTOR_BASE", "gs://mystique_qa/hudi-vector").rstrip("/")
# Point at a concrete Hudi table path (the folder containing .hoodie/)
TABLE_PATH = os.environ.get("HOODIE_VECTOR_TABLE_PATH", TABLE_BASE + "/vector_connect_it_demo").rstrip("/")

# Spark Connect (optional — leave None for gsutil-only inspection)
SPARK_REMOTE = "sc://lh-benchmark-hudi-vector.sample-app.spark.apps.useast4-dev-gke-st-spark00.k8s.us.walmart.net:443/;use_ssl=true"

PARQUET_VECTORIZED_READER = os.environ.get("HOODIE_VECTOR_TEST_PARQUET_VECTORIZED_READER", "false")
LOCK_PROVIDER = os.environ.get(
    "HOODIE_VECTOR_TEST_LOCK_PROVIDER",
    "org.apache.hudi.client.transaction.lock.InProcessLockProvider",
)
VECTOR_QUANTIZER = os.environ.get("HOODIE_VECTOR_TEST_QUANTIZER", "IVF_RABITQ")
MATERIALIZE_ON_CREATE = os.environ.get("HOODIE_VECTOR_TEST_MATERIALIZE_ON_CREATE", "false").lower() == "true"
VECTOR_NUM_ROWS = int(os.environ.get("HOODIE_VECTOR_TEST_NUM_ROWS", "1000000"))
VECTOR_SYNTHETIC_CLUSTERS = int(os.environ.get("HOODIE_VECTOR_SYNTHETIC_CLUSTERS", "128"))
VECTOR_QUERY_ROW_ID = int(os.environ.get("HOODIE_VECTOR_QUERY_ROW_ID", "12345"))
VECTOR_QUERY_TOPK = int(os.environ.get("HOODIE_VECTOR_QUERY_TOPK", "10"))

print("TABLE_PATH =", TABLE_PATH)
print("SPARK_REMOTE =", SPARK_REMOTE or "(not set — Spark cells will be skipped)")
print("PARQUET_VECTORIZED_READER =", PARQUET_VECTORIZED_READER)
print("LOCK_PROVIDER =", LOCK_PROVIDER)
print("VECTOR_QUANTIZER =", VECTOR_QUANTIZER)
print("MATERIALIZE_ON_CREATE =", MATERIALIZE_ON_CREATE)
print("VECTOR_NUM_ROWS =", VECTOR_NUM_ROWS)
print("VECTOR_SYNTHETIC_CLUSTERS =", VECTOR_SYNTHETIC_CLUSTERS)
print("VECTOR_QUERY_ROW_ID =", VECTOR_QUERY_ROW_ID)
print("VECTOR_QUERY_TOPK =", VECTOR_QUERY_TOPK)

TABLE_PATH = gs://mystique_qa/hudi-vector/vector_connect_it_demo
SPARK_REMOTE = sc://lh-benchmark-hudi-vector.sample-app.spark.apps.useast4-dev-gke-st-spark00.k8s.us.walmart.net:443/;use_ssl=true
PARQUET_VECTORIZED_READER = false
LOCK_PROVIDER = org.apache.hudi.client.transaction.lock.InProcessLockProvider
VECTOR_QUANTIZER = IVF_RABITQ
MATERIALIZE_ON_CREATE = False
VECTOR_NUM_ROWS = 1000000
VECTOR_SYNTHETIC_CLUSTERS = 128
VECTOR_QUERY_ROW_ID = 12345
VECTOR_QUERY_TOPK = 10


## 3. Live cluster note

From the driver pod you inspected:

- `/opt/hudi/hudi-spark4.0-bundle_2.13-1.2.0-vector-SNAPSHOT.jar`
- `/opt/hudi/hudi-azure-bundle-1.2.0-SNAPSHOT.jar`

That confirms the **Hudi jar exists on the driver filesystem**. The next question is whether the **running Spark Connect session** is actually using Hudi-related configs and classpath entries. The next cell prints the live session values before any table read.

In [2]:
# Live Spark Connect config probe
# This is the first cell to run before any Hudi read/write.

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

LIVE_CONFIG_KEYS = [
    "spark.serializer",
    "spark.kryo.registrator",
    "spark.sql.extensions",
    "spark.sql.catalog.spark_catalog",
    "spark.sql.parquet.enableVectorizedReader",
    "spark.jars",
    "spark.driver.extraClassPath",
    "spark.executor.extraClassPath",
    "spark.hadoop.fs.gs.impl",
    "spark.hadoop.fs.AbstractFileSystem.gs.impl",
]

if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")
    print("Spark version:", spark.version)
    print("Remote:", SPARK_REMOTE)
    print()
    print("=== spark.conf.get(...) ===")
    for key in LIVE_CONFIG_KEYS:
        try:
            value = spark.conf.get(key)
        except Exception as e:
            value = f"<unset: {type(e).__name__}: {e}>"
        print(f"{key} = {value}")

    print()
    print("=== spark.sql('SET -v') filtered ===")
    try:
        rows = spark.sql("SET -v").collect()
        matches = []
        for row in rows:
            key = getattr(row, "key", None)
            if key and (
                key.startswith("spark.serializer")
                or key.startswith("spark.kryo")
                or key.startswith("spark.sql.extensions")
                or key.startswith("spark.sql.catalog.spark_catalog")
                or key.startswith("spark.jars")
                or key.startswith("spark.driver.extraClassPath")
                or key.startswith("spark.executor.extraClassPath")
                or key.startswith("spark.hadoop.fs.gs")
                or key.startswith("spark.hadoop.fs.AbstractFileSystem.gs")
            ):
                matches.append(row)
        for row in matches:
            print(f"{row.key} = {row.value}")
    except Exception as e:
        print("SET -v failed:", e)
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

Spark version: 4.0.1
Remote: sc://lh-benchmark-hudi-vector.sample-app.spark.apps.useast4-dev-gke-st-spark00.k8s.us.walmart.net:443/;use_ssl=true

=== spark.conf.get(...) ===
spark.serializer = org.apache.spark.serializer.KryoSerializer
spark.kryo.registrator = org.apache.spark.HoodieSparkKryoRegistrar
spark.sql.extensions = org.apache.spark.sql.hudi.HoodieSparkSessionExtension
spark.sql.catalog.spark_catalog = org.apache.spark.sql.hudi.catalog.HoodieCatalog
spark.sql.parquet.enableVectorizedReader = false
spark.jars = file:/tmp/spark-ad2ef84a-b7c8-4b95-9794-046b6271c11a/spark-sql-kafka-0-10_2.13-4.0.1.jar,file:/tmp/spark-ad2ef84a-b7c8-4b95-9794-046b6271c11a/spark-token-provider-kafka-0-10_2.13-4.0.1.jar,file:/tmp/spark-ad2ef84a-b7c8-4b95-9794-046b6271c11a/kafka-clients-3.7.1.jar,file:/tmp/spark-ad2ef84a-b7c8-4b95-9794-046b6271c11a/commons-pool2-2.12.0.jar,file:/tmp/.ivy2.5.2/jars/com.google.cloud.bigdataoss_gcs-connector-hadoop3-2.2.26-shaded.jar,file:/tmp/.ivy2.5.2/jars/org.apache.had

## 3. What to look for in `hoodie.properties`

After running the print cell above, search the output for:

- **`hoodie.metadata.enable`** — metadata table must be enabled for MDT index partitions.
- **`hoodie.table.metadata.partitions`** — comma-separated list; look for **`vector_index_...`** entries when the vector index partition is registered.
- **`hoodie.table.metadata.partitions.inflight`** — partitions still building.

If you only see **`files`**, **`column_stats`**, etc., but no **`vector_index_`**, the IVF index partition may not be initialized yet (depends on your Hudi version and write pipeline).

**Data plane** (VECTOR column) still appears in the Parquet schema as **`ArrayType(FloatType)`** with Spark field metadata **`hudi_type=VECTOR(dim)`** — that is independent of whether the MDT vector index partition exists.

In [ ]:
# Optional: decode centroidBytes layout (K clusters × D dims × float32)
# Use when you have raw bytes (e.g. from debugging MDT); K = len(buf) // (D * 4).

import numpy as np


def decode_centroids_le(buf: bytes, dim: int) -> np.ndarray:
    """Return array of shape (K, dim) for little-endian float32 centroid blob."""
    n_floats = len(buf) // 4
    if n_floats * 4 != len(buf):
        raise ValueError("byte length must be multiple of 4")
    if n_floats % dim != 0:
        raise ValueError(f"n_floats={n_floats} not divisible by dim={dim}")
    flat = np.frombuffer(buf, dtype="<f4")
    k = n_floats // dim
    return flat.reshape(k, dim)


# Example: 3 clusters, 8 dimensions → 3 * 8 * 4 = 96 bytes
_demo = np.arange(3 * 8, dtype="<f4").tobytes()
print("demo centroids shape:", decode_centroids_le(_demo, dim=8).shape)
print(decode_centroids_le(_demo, dim=8))

## 5. Optional: Spark — read an existing Hudi table

Run the **live config probe** cell above first. This cell does **not** deserialize MDT Avro payloads, but it confirms the **embedding** column and **`hudi_type`** metadata on the dataset.

Requires: `pyspark`, cluster with Hudi + GCS, and a real Hudi table at `TABLE_PATH`. If `TABLE_PATH` does not exist yet, run the **build sample table** cell below first.

In [ ]:
HOODIE_TYPE_KEY = "hudi_type"  # HoodieSchema.TYPE_METADATA_FIELD
WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT
    print(f"SSL cert configured: {WALMART_CA_CERT}")
else:
    print(f"SSL cert not found at {WALMART_CA_CERT}")

if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")
    try:
        print("Attempting Hudi read from:", TABLE_PATH)
        df = spark.read.format("hudi").load(TABLE_PATH)
        print("rows:", df.count())
        df.printSchema()
        for f in df.schema.fields:
            if f.name == "embedding":
                print("embedding metadata:", dict(f.metadata))
    except Exception as e:
        print("Hudi read failed:", type(e).__name__, e)
        print("Hint: verify the live config probe shows Hudi extensions/catalog, and that TABLE_PATH points to an existing Hudi table.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

## 6. Build a sample Hudi vector table

This cell creates a small deterministic Hudi table at `TABLE_PATH` so the later read and search cells have a concrete dataset to work with.

Dataset shape:

- 9 rows
- 3 logical clusters: `west`, `mid`, `east`
- `embedding` stored as `ARRAY<FLOAT>` with Spark field metadata `hudi_type=VECTOR(8)`
- write mode: `COPY_ON_WRITE`

After this runs, rerun the `hoodie.properties` and metadata tree cells above to inspect the table layout.

In [ ]:
# Build a small Hudi table with deterministic VECTOR(8) embeddings.

import json

HOODIE_TYPE_KEY = "hudi_type"
WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

if SPARK_REMOTE:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import ArrayType, FloatType, StringType, StructField, StructType

    dim = 8
    schema = StructType([
        StructField("id", StringType(), False),
        StructField("label", StringType(), False),
        StructField("embedding", ArrayType(FloatType(), False), False, metadata={HOODIE_TYPE_KEY: f"VECTOR({dim})"}),
    ])

    def vector_write_schema_json(d: int) -> str:
        return json.dumps({
            "type": "record",
            "name": "vector_connect_record",
            "namespace": "hoodie.vector.connect",
            "fields": [
                {"name": "id", "type": "string"},
                {"name": "label", "type": "string"},
                {
                    "name": "embedding",
                    "type": {
                        "type": "fixed",
                        "name": f"vector_float_{d}",
                        "size": d * 4,
                        "logicalType": "vector",
                        "dimension": d,
                        "elementType": "FLOAT",
                        "storageBacking": "FIXED_BYTES",
                    },
                },
            ],
        })

    def vec(x: float):
        return [float(x)] + [0.0] * (dim - 1)

    rows = [
        ("west-0", "west", vec(0.0)),
        ("west-1", "west", vec(0.4)),
        ("west-2", "west", vec(0.8)),
        ("mid-0", "mid", vec(10.0)),
        ("mid-1", "mid", vec(10.3)),
        ("mid-2", "mid", vec(9.7)),
        ("east-0", "east", vec(20.0)),
        ("east-1", "east", vec(20.4)),
        ("east-2", "east", vec(19.6)),
    ]

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")
    try:
        df = spark.createDataFrame(rows, schema=schema)
        (
            df.write.format("hudi")
            .mode("overwrite")
            .option("hoodie.datasource.write.recordkey.field", "id")
            .option("hoodie.datasource.write.precombine.field", "id")
            .option("hoodie.datasource.write.table.name", "vector_connect_it_demo")
            .option("hoodie.datasource.write.table.type", "COPY_ON_WRITE")
            .option("hoodie.write.schema", vector_write_schema_json(dim))
            .option("hoodie.metadata.enable", "true")
            .option("hoodie.write.lock.provider", LOCK_PROVIDER)
            .option("path", TABLE_PATH)
            .save()
        )
        print(f"Built sample Hudi vector table at: {TABLE_PATH}")
        print("Rows written:", len(rows))
        print("Labels:", sorted({r[1] for r in rows}))
    except Exception as e:
        print("Build failed:", type(e).__name__, e)
        print("Hint: rerun the live config probe first and confirm the session is Hudi-enabled.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

## 6b. MDT append regression check

This cell performs one small follow-up `append` write after table creation.

Why this matters:
- the latest fix is in the **metadata-table append/update path**
- recent failures happened while updating the MDT `files` partition, not during the initial table bootstrap
- if this cell passes, the notebook has already exercised the same class of MDT update that was failing for RecordLevelIndex and streaming writes

In [ ]:
# Append one extra row to exercise the MDT files-partition update path.

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")
    try:
        before_count = spark.read.format("hudi").load(TABLE_PATH).count()
        append_rows = [("post-create-0", "post-create", vec(30.0))]
        append_df = spark.createDataFrame(append_rows, schema=schema)
        (
            append_df.write.format("hudi")
            .mode("append")
            .option("hoodie.datasource.write.recordkey.field", "id")
            .option("hoodie.datasource.write.precombine.field", "id")
            .option("hoodie.datasource.write.table.name", "vector_connect_it_demo")
            .option("hoodie.datasource.write.table.type", "COPY_ON_WRITE")
            .option("hoodie.write.schema", vector_write_schema_json(dim))
            .option("hoodie.metadata.enable", "true")
            .option("hoodie.write.lock.provider", LOCK_PROVIDER)
            .option("path", TABLE_PATH)
            .save()
        )
        after_df = spark.read.format("hudi").load(TABLE_PATH)
        after_count = after_df.count()
        new_row = after_df.where("id = 'post-create-0'").select("id", "label").collect()

        print("Append write finished.")
        print("Row count before:", before_count)
        print("Row count after:", after_count)
        print("New row sample:", new_row)

        assert after_count == before_count + len(append_rows), (
            f"expected {before_count + len(append_rows)} rows after append, got {after_count}"
        )
        assert new_row, "expected appended row to be readable after MDT update"
        print("MDT append/update regression check passed.")
    except Exception as e:
        print("Append regression check failed:", type(e).__name__, e)
        raise
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

## 7. Explicit vector-index build attempt

This cell rewrites the sample table using the **RFC draft** vector-index writer options:

- `hoodie.metadata.vector.index.enable`
- `hoodie.metadata.vector.index.column`
- `hoodie.metadata.vector.index.dimension`
- `hoodie.metadata.vector.index.metric`
- `hoodie.metadata.vector.index.algorithm`
- `hoodie.metadata.vector.index.num_clusters`

Important: these keys are documented in `rfc/rfc-103/rfc-103.md`, but they are **not currently discoverable in the implementation code** in this branch. So this cell may do one of three things:

1. fail fast,
2. succeed but ignore the options,
3. succeed and actually create vector-index MDT artifacts.

The next verification cell checks the real artifact: a `vector_index_<name>` metadata partition.

In [ ]:
# Attempt an explicit vector-index build using the RFC draft datasource options.

import json

VECTOR_INDEX_NAME = "embedding_idx"
VECTOR_INDEX_PARTITION = f"vector_index_{VECTOR_INDEX_NAME}"

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

if SPARK_REMOTE:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import ArrayType, FloatType, StringType, StructField, StructType

    dim = 8
    schema = StructType([
        StructField("id", StringType(), False),
        StructField("label", StringType(), False),
        StructField("embedding", ArrayType(FloatType(), False), False, metadata={"hudi_type": f"VECTOR({dim})"}),
    ])

    def vector_write_schema_json(d: int) -> str:
        return json.dumps({
            "type": "record",
            "name": "vector_connect_record",
            "namespace": "hoodie.vector.connect",
            "fields": [
                {"name": "id", "type": "string"},
                {"name": "label", "type": "string"},
                {
                    "name": "embedding",
                    "type": {
                        "type": "fixed",
                        "name": f"vector_float_{d}",
                        "size": d * 4,
                        "logicalType": "vector",
                        "dimension": d,
                        "elementType": "FLOAT",
                        "storageBacking": "FIXED_BYTES",
                    },
                },
            ],
        })

    def vec(x: float):
        return [float(x)] + [0.0] * (dim - 1)

    rows = [
        ("west-0", "west", vec(0.0)),
        ("west-1", "west", vec(0.4)),
        ("west-2", "west", vec(0.8)),
        ("mid-0", "mid", vec(10.0)),
        ("mid-1", "mid", vec(10.3)),
        ("mid-2", "mid", vec(9.7)),
        ("east-0", "east", vec(20.0)),
        ("east-1", "east", vec(20.4)),
        ("east-2", "east", vec(19.6)),
    ]

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")
    try:
        df = spark.createDataFrame(rows, schema=schema)
        (
            df.write.format("hudi")
            .mode("overwrite")
            .option("hoodie.datasource.write.recordkey.field", "id")
            .option("hoodie.datasource.write.precombine.field", "id")
            .option("hoodie.datasource.write.table.name", "vector_connect_it_demo")
            .option("hoodie.datasource.write.table.type", "COPY_ON_WRITE")
            .option("hoodie.write.schema", vector_write_schema_json(dim))
            .option("hoodie.metadata.enable", "true")
            .option("hoodie.metadata.vector.index.enable", "true")
            .option("hoodie.metadata.vector.index.name", VECTOR_INDEX_NAME)
            .option("hoodie.metadata.vector.index.column", "embedding")
            .option("hoodie.metadata.vector.index.dimension", str(dim))
            .option("hoodie.metadata.vector.index.metric", "l2")
            .option("hoodie.metadata.vector.index.algorithm", "ivfflat")
            .option("hoodie.metadata.vector.index.num_clusters", "3")
            .option("path", TABLE_PATH)
            .save()
        )
        print("Explicit vector-index build attempt finished.")
        print("Requested index name:", VECTOR_INDEX_NAME)
        print("Expected MDT partition:", VECTOR_INDEX_PARTITION)
    except Exception as e:
        print("Explicit vector-index build failed:", type(e).__name__, e)
        print("This usually means the session is not Hudi-enabled, or the branch does not yet implement the RFC vector-index writer options.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

In [ ]:
# Explicit SQL vector-index creation on the existing Hudi table.
# Run this after the sample table has been written.

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

CATALOG_TABLE = os.environ.get("HOODIE_VECTOR_CATALOG_TABLE", "vector_connect_it_demo_sql")
VECTOR_INDEX_NAME = os.environ.get("HOODIE_VECTOR_INDEX_NAME", "embedding_idx")
VECTOR_DIMENSION = int(os.environ.get("HOODIE_VECTOR_DIMENSION", "8"))
VECTOR_NUM_CLUSTERS = int(os.environ.get("HOODIE_VECTOR_NUM_CLUSTERS", "3"))
VECTOR_METRIC = os.environ.get("HOODIE_VECTOR_METRIC", "l2")
VECTOR_ALGORITHM = os.environ.get("HOODIE_VECTOR_ALGORITHM", "ivfflat")

if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")
    try:
        print("Registering catalog table:", CATALOG_TABLE)
        spark.sql(f"DROP TABLE IF EXISTS {CATALOG_TABLE}")
        spark.sql(f"CREATE TABLE {CATALOG_TABLE} USING hudi LOCATION '{TABLE_PATH}'")

        create_index_options = [
            f"'vector.dimension' = '{VECTOR_DIMENSION}'",
            f"'vector.num_clusters' = '{VECTOR_NUM_CLUSTERS}'",
            f"'vector.metric' = '{VECTOR_METRIC}'",
            f"'vector.algorithm' = '{VECTOR_ALGORITHM}'",
        ]
        if VECTOR_QUANTIZER:
            create_index_options.append(f"'vector.quantizer' = '{VECTOR_QUANTIZER}'")
        if MATERIALIZE_ON_CREATE:
            create_index_options.append("'vector.rabitq.materialize.on.create' = 'true'")

        create_index_sql = f"""
        CREATE INDEX {VECTOR_INDEX_NAME}
        ON {CATALOG_TABLE}
        USING VECTOR (embedding)
        OPTIONS (
          {', '.join(create_index_options)}
        )
        """

        print("Running SQL vector index creation:")
        print(create_index_sql)
        spark.sql(create_index_sql)

        print("SHOW INDEXES output:")
        spark.sql(f"SHOW INDEXES FROM {CATALOG_TABLE}").show(truncate=False)
        print("Explicit SQL CREATE INDEX finished.")
    except Exception as e:
        print("Explicit SQL CREATE INDEX failed:", type(e).__name__, e)
        raise
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

## 8. Verify vector-index artifacts

This is the real test for the explicit build above.

It checks two things:

- whether `hoodie.properties` declares a `vector_index_<name>` metadata partition
- whether files exist under `.hoodie/metadata/vector_index_<name>/`

If both checks fail, then the RFC options were ignored or the explicit vector-index builder is not implemented in this branch.

## 9. MDT vector metadata samples

This section prints concrete rows from `hudi_metadata(...)` for the vector index partition:

- the centroid record
- the quantizer record (`IVF_RABITQ` seed / code width / normalization mode)
- assignment samples (`record_key -> cluster_id + file_group_id + partition_path`)
- `fg_mapping` samples (`cluster_id + partition_path -> file_group_ids`)

These are the most direct way to validate that the index build wrote the expected MDT structures.

In [ ]:
VECTOR_RECORD_TYPE = 8
VECTOR_FG_KEY_PREFIX = "__fg__/"
VECTOR_CENTROIDS_KEY = "__centroids__"
VECTOR_QUANTIZER_KEY = "__quantizer__"

if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")

    centroid_df = spark.sql(
        f"""
        SELECT
          key,
          VectorIndexMetadata.clusterId AS cluster_id,
          length(VectorIndexMetadata.centroidBytes) AS centroid_bytes
        FROM hudi_metadata('{TABLE_PATH}')
        WHERE type = {VECTOR_RECORD_TYPE}
          AND key = '{VECTOR_CENTROIDS_KEY}'
        """
    )
    print("Centroid record:")
    centroid_df.show(truncate=False)

    quantizer_df = spark.sql(
        f"""
        SELECT
          key,
          VectorIndexMetadata.quantizerType AS quantizer_type,
          VectorIndexMetadata.quantizedCodeBytes AS quantized_code_bytes,
          VectorIndexMetadata.randomSeed AS random_seed,
          VectorIndexMetadata.assumeNormalized AS assume_normalized
        FROM hudi_metadata('{TABLE_PATH}')
        WHERE type = {VECTOR_RECORD_TYPE}
          AND key = '{VECTOR_QUANTIZER_KEY}'
        """
    )
    print("Quantizer metadata:")
    quantizer_df.show(truncate=False)

    assignments_df = spark.sql(
        f"""
        SELECT
          key,
          VectorIndexMetadata.clusterId AS cluster_id,
          VectorIndexMetadata.fileGroupId AS file_group_id,
          VectorIndexMetadata.partitionPath AS partition_path,
          VectorIndexMetadata.isDeleted AS is_deleted
        FROM hudi_metadata('{TABLE_PATH}')
        WHERE type = {VECTOR_RECORD_TYPE}
          AND key <> '{VECTOR_CENTROIDS_KEY}'
          AND key <> '{VECTOR_QUANTIZER_KEY}'
          AND key NOT LIKE '{VECTOR_FG_KEY_PREFIX}%'
        ORDER BY key
        LIMIT 10
        """
    )
    print("Assignment samples:")
    assignments_df.show(truncate=False)

    fg_mapping_df = spark.sql(
        f"""
        SELECT
          key,
          VectorIndexMetadata.clusterId AS cluster_id,
          VectorIndexMetadata.partitionPath AS partition_path,
          VectorIndexMetadata.fileGroupIds AS file_group_ids,
          VectorIndexMetadata.vectorCount AS vector_count,
          VectorIndexMetadata.lastUpdatedTs AS last_updated_ts
        FROM hudi_metadata('{TABLE_PATH}')
        WHERE type = {VECTOR_RECORD_TYPE}
          AND key LIKE '{VECTOR_FG_KEY_PREFIX}%'
        ORDER BY key
        LIMIT 10
        """
    )
    print("fg_mapping samples:")
    fg_mapping_df.show(truncate=False)
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

In [ ]:
# Verify whether the explicit vector-index build actually produced MDT artifacts.

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")
    properties_path = f"{TABLE_PATH}/.hoodie/hoodie.properties"
    metadata_glob = f"{TABLE_PATH}/.hoodie/metadata/{VECTOR_INDEX_PARTITION}/*"

    partition_declared = False
    artifact_count = 0

    try:
        props_df = spark.read.text(properties_path)
        prop_lines = [r.value for r in props_df.collect()]
        interesting = [line for line in prop_lines if "metadata" in line or "vector" in line]
        print("Relevant hoodie.properties lines:")
        for line in interesting:
            print(line)
        partition_declared = any(VECTOR_INDEX_PARTITION in line for line in prop_lines)
    except Exception as e:
        print("Could not read hoodie.properties via Spark:", type(e).__name__, e)

    try:
        files_df = spark.read.format("binaryFile").load(metadata_glob)
        artifact_count = files_df.count()
        print(f"Files under {metadata_glob}: {artifact_count}")
    except Exception as e:
        print("Could not list vector-index metadata artifacts via Spark:", type(e).__name__, e)

    print("partition_declared =", partition_declared)
    print("artifact_count =", artifact_count)

    assert partition_declared or artifact_count > 0, (
        f"No explicit vector-index artifacts found for {VECTOR_INDEX_PARTITION}. "
        "The RFC option keys were likely ignored, or the builder path is not implemented in this branch."
    )
    print("Vector-index artifact verification passed.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

## 9. Post-index RaBitQ insert

This section writes **brand-new vectors after the index already exists**.

That is the cleanest way to validate the RaBitQ write path right now:

- the fresh rows should be assigned into the MDT vector index
- the fresh base files should carry the hidden RaBitQ columns
- we can then score only those fresh rows and verify the approximate ranking behavior

This avoids depending on `vector.rabitq.materialize.on.create=true` for older rows.

In [ ]:
# Append brand-new rows after CREATE INDEX so the write path itself has to
# derive the hidden RaBitQ columns for those rows.

import json

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

VECTOR_INDEX_NAME = os.environ.get("HOODIE_VECTOR_INDEX_NAME", "embedding_idx")
RABITQ_INSERT_LABEL = os.environ.get("HOODIE_VECTOR_RABITQ_INSERT_LABEL", "rabitq-fresh")
RABITQ_INSERT_VALUES_ENV = os.environ.get("HOODIE_VECTOR_RABITQ_INSERT_VALUES")
VECTOR_DIMENSION = int(os.environ.get("HOODIE_VECTOR_DIMENSION", "8"))
VECTOR_TABLE_NAME = os.environ.get("HOODIE_VECTOR_TABLE_NAME", "vector_connect_it_demo")


def parse_insert_values():
    if RABITQ_INSERT_VALUES_ENV:
        return [float(value.strip()) for value in RABITQ_INSERT_VALUES_ENV.split(",") if value.strip()]
    return [
        10.60,
        10.28,
        10.18,
        10.12,
        10.08,
        10.03,
        9.98,
        9.92,
        9.80,
        9.55,
        8.90,
        11.40,
    ]


if SPARK_REMOTE:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import ArrayType, FloatType, StringType, StructField, StructType

    def vector_write_schema_json(d: int) -> str:
        return json.dumps({
            "type": "record",
            "name": "vector_connect_record",
            "namespace": "hoodie.vector.connect",
            "fields": [
                {"name": "id", "type": "string"},
                {"name": "label", "type": "string"},
                {
                    "name": "embedding",
                    "type": {
                        "type": "fixed",
                        "name": f"vector_float_{d}",
                        "size": d * 4,
                        "logicalType": "vector",
                        "dimension": d,
                        "elementType": "FLOAT",
                        "storageBacking": "FIXED_BYTES",
                    },
                },
            ],
        })

    def vec(x: float):
        return [float(x)] + [0.0] * (VECTOR_DIMENSION - 1)

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")

    insert_values = parse_insert_values()
    insert_rows = [
        (f"{RABITQ_INSERT_LABEL}-{idx:03d}", RABITQ_INSERT_LABEL, vec(value))
        for idx, value in enumerate(insert_values)
    ]
    insert_ids = [row[0] for row in insert_rows]
    ids_sql = ", ".join("'" + value.replace("'", "''") + "'" for value in insert_ids)

    schema = StructType([
        StructField("id", StringType(), False),
        StructField("label", StringType(), False),
        StructField("embedding", ArrayType(FloatType(), False), False, metadata={HOODIE_TYPE_KEY: f"VECTOR({VECTOR_DIMENSION})"}),
    ])

    append_df = spark.createDataFrame(insert_rows, schema=schema)
    (
        append_df.write.format("hudi")
        .mode("append")
        .option("hoodie.datasource.write.recordkey.field", "id")
        .option("hoodie.datasource.write.precombine.field", "id")
        .option("hoodie.datasource.write.table.name", VECTOR_TABLE_NAME)
        .option("hoodie.datasource.write.table.type", "COPY_ON_WRITE")
        .option("hoodie.write.schema", vector_write_schema_json(VECTOR_DIMENSION))
        .option("hoodie.metadata.enable", "true")
        .option("hoodie.write.lock.provider", LOCK_PROVIDER)
        .option("path", TABLE_PATH)
        .save()
    )

    fresh_rows = (
        spark.read.format("hudi")
        .load(TABLE_PATH)
        .where(f"label = '{RABITQ_INSERT_LABEL}'")
        .select("id", "label")
        .orderBy("id")
        .collect()
    )
    assignment_count = spark.sql(
        f"""
        SELECT count(*) AS cnt
        FROM hudi_metadata('{TABLE_PATH}')
        WHERE type = 8
          AND key IN ({ids_sql})
        """
    ).collect()[0].cnt

    print("Inserted fresh post-index rows for RaBitQ validation.")
    print("insert_label =", RABITQ_INSERT_LABEL)
    print("insert_count =", len(insert_rows))
    print("insert_values =", insert_values)
    print("fresh_rows_visible_via_hudi =", len(fresh_rows))
    print("mdt_assignment_count_for_fresh_rows =", assignment_count)
    print("fresh_row_ids =", [row.id for row in fresh_rows])

    assert len(fresh_rows) == len(insert_rows), (
        f"Expected {len(insert_rows)} fresh rows through the Hudi read path, got {len(fresh_rows)}"
    )
    assert assignment_count == len(insert_rows), (
        f"Expected {len(insert_rows)} MDT assignment records for the fresh rows, got {assignment_count}"
    )
    print("Fresh-row insert + MDT assignment verification passed.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

## 8b. Isolated backfill-on-create RaBitQ test

This section creates a **brand-new table**, then runs `CREATE INDEX ... USING VECTOR` with `vector.rabitq.materialize.on.create = true`.

The goal is to test the specific case you asked about:

- all rows already exist before index creation
- index creation should backfill hidden RaBitQ columns for those existing rows
- verification reads the raw Parquet files directly to check whether `_hudi_vec_*` columns were physically written

This keeps the result isolated from the earlier table state and from any partial post-index append behavior.

In [ ]:
# Create a brand-new table and ask CREATE INDEX to backfill RaBitQ hidden
# columns for the already-existing rows.

import json

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

BACKFILL_TABLE_PATH = os.environ.get(
    "HOODIE_VECTOR_BACKFILL_TABLE_PATH",
    TABLE_BASE + "/vector_connect_it_demo_rabitq_backfill",
).rstrip("/")
BACKFILL_CATALOG_TABLE = os.environ.get(
    "HOODIE_VECTOR_BACKFILL_CATALOG_TABLE",
    "vector_connect_it_demo_rabitq_backfill_sql",
)
BACKFILL_INDEX_NAME = os.environ.get("HOODIE_VECTOR_BACKFILL_INDEX_NAME", "embedding_idx")
BACKFILL_DIMENSION = int(os.environ.get("HOODIE_VECTOR_DIMENSION", "8"))
BACKFILL_NUM_CLUSTERS = int(os.environ.get("HOODIE_VECTOR_NUM_CLUSTERS", "3"))
BACKFILL_METRIC = os.environ.get("HOODIE_VECTOR_METRIC", "l2")
BACKFILL_ALGORITHM = os.environ.get("HOODIE_VECTOR_ALGORITHM", "ivfflat")
BACKFILL_TABLE_NAME = os.environ.get("HOODIE_VECTOR_BACKFILL_TABLE_NAME", "vector_connect_it_demo_rabitq_backfill")


if SPARK_REMOTE:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import ArrayType, FloatType, StringType, StructField, StructType

    def vector_write_schema_json(d: int) -> str:
        return json.dumps({
            "type": "record",
            "name": "vector_connect_record",
            "namespace": "hoodie.vector.connect",
            "fields": [
                {"name": "id", "type": "string"},
                {"name": "label", "type": "string"},
                {
                    "name": "embedding",
                    "type": {
                        "type": "fixed",
                        "name": f"vector_float_{d}",
                        "size": d * 4,
                        "logicalType": "vector",
                        "dimension": d,
                        "elementType": "FLOAT",
                        "storageBacking": "FIXED_BYTES",
                    },
                },
            ],
        })

    def vec(x: float):
        return [float(x)] + [0.0] * (BACKFILL_DIMENSION - 1)

    rows = [
        ("west-0", "west", vec(0.0)),
        ("west-1", "west", vec(0.4)),
        ("west-2", "west", vec(0.8)),
        ("mid-0", "mid", vec(10.0)),
        ("mid-1", "mid", vec(10.3)),
        ("mid-2", "mid", vec(9.7)),
        ("east-0", "east", vec(20.0)),
        ("east-1", "east", vec(20.4)),
        ("east-2", "east", vec(19.6)),
    ]

    schema = StructType([
        StructField("id", StringType(), False),
        StructField("label", StringType(), False),
        StructField("embedding", ArrayType(FloatType(), False), False, metadata={HOODIE_TYPE_KEY: f"VECTOR({BACKFILL_DIMENSION})"}),
    ])

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")

    base_df = spark.createDataFrame(rows, schema=schema)
    (
        base_df.write.format("hudi")
        .mode("overwrite")
        .option("hoodie.datasource.write.recordkey.field", "id")
        .option("hoodie.datasource.write.precombine.field", "id")
        .option("hoodie.datasource.write.table.name", BACKFILL_TABLE_NAME)
        .option("hoodie.datasource.write.table.type", "COPY_ON_WRITE")
        .option("hoodie.write.schema", vector_write_schema_json(BACKFILL_DIMENSION))
        .option("hoodie.metadata.enable", "true")
        .option("hoodie.write.lock.provider", LOCK_PROVIDER)
        .option("path", BACKFILL_TABLE_PATH)
        .save()
    )

    spark.sql(f"DROP TABLE IF EXISTS {BACKFILL_CATALOG_TABLE}")
    spark.sql(f"CREATE TABLE {BACKFILL_CATALOG_TABLE} USING hudi LOCATION '{BACKFILL_TABLE_PATH}'")

    create_index_options = [
        f"'vector.dimension' = '{BACKFILL_DIMENSION}'",
        f"'vector.num_clusters' = '{BACKFILL_NUM_CLUSTERS}'",
        f"'vector.metric' = '{BACKFILL_METRIC}'",
        f"'vector.algorithm' = '{BACKFILL_ALGORITHM}'",
        "'vector.rabitq.materialize.on.create' = 'true'",
    ]
    if VECTOR_QUANTIZER:
        create_index_options.append(f"'vector.quantizer' = '{VECTOR_QUANTIZER}'")

    create_index_sql = f"""
    CREATE INDEX {BACKFILL_INDEX_NAME}
    ON {BACKFILL_CATALOG_TABLE}
    USING VECTOR (embedding)
    OPTIONS (
      {', '.join(create_index_options)}
    )
    """

    print("BACKFILL_TABLE_PATH =", BACKFILL_TABLE_PATH)
    print("BACKFILL_CATALOG_TABLE =", BACKFILL_CATALOG_TABLE)
    print("Running CREATE INDEX with RaBitQ backfill enabled:")
    print(create_index_sql)
    spark.sql(create_index_sql)
    print("SHOW INDEXES output:")
    spark.sql(f"SHOW INDEXES FROM {BACKFILL_CATALOG_TABLE}").show(truncate=False)
    print("Backfill-on-create table setup finished.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

In [ ]:
# Verify whether CREATE INDEX with backfill enabled actually wrote the hidden
# RaBitQ columns for the rows that existed before index creation.

from pyspark.sql.functions import col, expr, length


if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    def logical_index_name(index_name):
        return index_name.removeprefix("vector_index_")

    def rabitq_binary_column(index_name):
        return f"_hudi_vec_{logical_index_name(index_name)}_binary_code"

    def rabitq_scalar_column(index_name):
        return f"_hudi_vec_{logical_index_name(index_name)}_scalar"

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")

    binary_col = rabitq_binary_column(BACKFILL_INDEX_NAME)
    scalar_col = rabitq_scalar_column(BACKFILL_INDEX_NAME)
    raw_df = (
        spark.read.format("parquet")
        .option("mergeSchema", "true")
        .load(os.environ.get("HOODIE_VECTOR_BACKFILL_RAW_DATA_GLOB", f"{BACKFILL_TABLE_PATH}/*.parquet"))
        .cache()
    )
    row_count = raw_df.count()
    available_columns = sorted(raw_df.columns)
    quantizer_row = spark.sql(
        f"""
        SELECT
          VectorIndexMetadata.quantizerType AS quantizer_type,
          VectorIndexMetadata.quantizedCodeBytes AS quantized_code_bytes,
          VectorIndexMetadata.randomSeed AS random_seed,
          VectorIndexMetadata.assumeNormalized AS assume_normalized
        FROM hudi_metadata('{BACKFILL_TABLE_PATH}')
        WHERE type = 8
          AND key = '__quantizer__'
        """
    ).collect()[0]

    print("BACKFILL_TABLE_PATH =", BACKFILL_TABLE_PATH)
    print("row_count =", row_count)
    print("visible_columns =", available_columns)
    print("quantizer_row =", quantizer_row)

    assert row_count > 0, "No rows found in the backfill test table"
    assert binary_col in raw_df.columns, (
        f"Expected backfilled RaBitQ binary column {binary_col} in raw Parquet schema"
    )
    if not bool(quantizer_row.assume_normalized):
        assert scalar_col in raw_df.columns, (
            f"Expected backfilled RaBitQ scalar column {scalar_col} in raw Parquet schema"
        )

    stats_exprs = [
        expr("count(*)").alias("rows_total"),
        expr(f"sum(case when {binary_col} is not null then 1 else 0 end)").alias("rows_with_binary"),
        expr(f"sum(case when length({binary_col}) > 0 then 1 else 0 end)").alias("rows_with_nonempty_binary"),
    ]
    if scalar_col in raw_df.columns:
        stats_exprs.append(expr(f"sum(case when {scalar_col} is not null then 1 else 0 end)").alias("rows_with_scalar"))
    hidden_stats = raw_df.select(*stats_exprs).collect()[0]

    print("hidden_column_stats =", hidden_stats)
    print("Hidden-column samples:")
    sample_columns = ["id", length(col(binary_col)).alias("binary_code_bytes")]
    if scalar_col in raw_df.columns:
        sample_columns.append(col(scalar_col).alias("scalar"))
    raw_df.select(*sample_columns).orderBy("id").show(truncate=False)

    assert hidden_stats.rows_with_binary == hidden_stats.rows_total, (
        f"Expected every pre-existing row to be backfilled with {binary_col}; got {hidden_stats.rows_with_binary}/{hidden_stats.rows_total}"
    )
    assert hidden_stats.rows_with_nonempty_binary == hidden_stats.rows_total, (
        f"Expected every pre-existing row to have a non-empty RaBitQ binary code; got {hidden_stats.rows_with_nonempty_binary}/{hidden_stats.rows_total}"
    )
    if scalar_col in raw_df.columns:
        assert hidden_stats.rows_with_scalar == hidden_stats.rows_total, (
            f"Expected every pre-existing row to be backfilled with {scalar_col}; got {hidden_stats.rows_with_scalar}/{hidden_stats.rows_total}"
        )

    print("Backfill-on-create RaBitQ verification passed.")
    raw_df.unpersist()
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

## 10. Verify hidden RaBitQ columns on fresh rows

This section inspects the **raw Parquet schema** for only the rows inserted in the previous step.

That gives a direct answer to the write-path question:

- did Hudi materialize `_hudi_vec_*_binary_code`?
- did it also materialize `_hudi_vec_*_scalar` when normalization is not assumed?
- are the fresh rows carrying non-null values for those hidden columns?

The raw Parquet read is intentional here because it validates physical storage of the RaBitQ artifacts even if the current Hudi read schema does not project them yet.

## 8c. Fresh unique-path backfill validation

Use this section when you want a completely new test table on every run.

It differs from the earlier backfill section in two ways:

- the table path, catalog table, and row ids are derived from a unique run id
- verification reads raw Parquet and keeps only the latest row version per `_hoodie_record_key`, so historical base files from the rewrite do not double-count rows

This is the cleanest validation path for `vector.rabitq.materialize.on.create = true`.

In [ ]:
# Create a brand-new uniquely named table, then run CREATE INDEX with
# RaBitQ backfill enabled for the existing rows.

import json
import uuid

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

BACKFILL_UNIQUE_RUN_ID = globals().get("BACKFILL_UNIQUE_RUN_ID") or os.environ.get("HOODIE_VECTOR_BACKFILL_UNIQUE_RUN_ID") or uuid.uuid4().hex[:8]
BACKFILL_UNIQUE_TABLE_BASENAME = os.environ.get(
    "HOODIE_VECTOR_BACKFILL_UNIQUE_TABLE_BASENAME",
    f"vector_rabitq_backfill_{BACKFILL_UNIQUE_RUN_ID}",
)
BACKFILL_UNIQUE_TABLE_PATH = os.environ.get(
    "HOODIE_VECTOR_BACKFILL_UNIQUE_TABLE_PATH",
    TABLE_BASE + f"/{BACKFILL_UNIQUE_TABLE_BASENAME}",
).rstrip("/")
BACKFILL_UNIQUE_CATALOG_TABLE = os.environ.get(
    "HOODIE_VECTOR_BACKFILL_UNIQUE_CATALOG_TABLE",
    f"{BACKFILL_UNIQUE_TABLE_BASENAME}_sql",
)
BACKFILL_UNIQUE_INDEX_NAME = os.environ.get("HOODIE_VECTOR_BACKFILL_UNIQUE_INDEX_NAME", "embedding_idx")
BACKFILL_UNIQUE_TABLE_NAME = os.environ.get("HOODIE_VECTOR_BACKFILL_UNIQUE_TABLE_NAME", BACKFILL_UNIQUE_TABLE_BASENAME)
BACKFILL_UNIQUE_DIMENSION = int(os.environ.get("HOODIE_VECTOR_DIMENSION", "8"))
BACKFILL_UNIQUE_NUM_CLUSTERS = int(os.environ.get("HOODIE_VECTOR_NUM_CLUSTERS", "3"))
BACKFILL_UNIQUE_METRIC = os.environ.get("HOODIE_VECTOR_METRIC", "l2")
BACKFILL_UNIQUE_ALGORITHM = os.environ.get("HOODIE_VECTOR_ALGORITHM", "ivfflat")

if SPARK_REMOTE:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import ArrayType, FloatType, StringType, StructField, StructType

    def vector_write_schema_json(d: int) -> str:
        return json.dumps({
            "type": "record",
            "name": "vector_connect_record",
            "namespace": "hoodie.vector.connect",
            "fields": [
                {"name": "id", "type": "string"},
                {"name": "label", "type": "string"},
                {
                    "name": "embedding",
                    "type": {
                        "type": "fixed",
                        "name": f"vector_float_{d}",
                        "size": d * 4,
                        "logicalType": "vector",
                        "dimension": d,
                        "elementType": "FLOAT",
                        "storageBacking": "FIXED_BYTES",
                    },
                },
            ],
        })

    def vec(x: float):
        return [float(x)] + [0.0] * (BACKFILL_UNIQUE_DIMENSION - 1)

    rows = [
        (f"bf-{BACKFILL_UNIQUE_RUN_ID}-west-0", "west", vec(0.0)),
        (f"bf-{BACKFILL_UNIQUE_RUN_ID}-west-1", "west", vec(0.4)),
        (f"bf-{BACKFILL_UNIQUE_RUN_ID}-west-2", "west", vec(0.8)),
        (f"bf-{BACKFILL_UNIQUE_RUN_ID}-mid-0", "mid", vec(10.0)),
        (f"bf-{BACKFILL_UNIQUE_RUN_ID}-mid-1", "mid", vec(10.3)),
        (f"bf-{BACKFILL_UNIQUE_RUN_ID}-mid-2", "mid", vec(9.7)),
        (f"bf-{BACKFILL_UNIQUE_RUN_ID}-east-0", "east", vec(20.0)),
        (f"bf-{BACKFILL_UNIQUE_RUN_ID}-east-1", "east", vec(20.4)),
        (f"bf-{BACKFILL_UNIQUE_RUN_ID}-east-2", "east", vec(19.6)),
    ]

    schema = StructType([
        StructField("id", StringType(), False),
        StructField("label", StringType(), False),
        StructField("embedding", ArrayType(FloatType(), False), False, metadata={HOODIE_TYPE_KEY: f"VECTOR({BACKFILL_UNIQUE_DIMENSION})"}),
    ])

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")

    base_df = spark.createDataFrame(rows, schema=schema)
    (
        base_df.write.format("hudi")
        .mode("overwrite")
        .option("hoodie.datasource.write.recordkey.field", "id")
        .option("hoodie.datasource.write.precombine.field", "id")
        .option("hoodie.datasource.write.table.name", BACKFILL_UNIQUE_TABLE_NAME)
        .option("hoodie.datasource.write.table.type", "COPY_ON_WRITE")
        .option("hoodie.write.schema", vector_write_schema_json(BACKFILL_UNIQUE_DIMENSION))
        .option("hoodie.metadata.enable", "true")
        .option("hoodie.write.lock.provider", LOCK_PROVIDER)
        .option("path", BACKFILL_UNIQUE_TABLE_PATH)
        .save()
    )

    spark.sql(f"DROP TABLE IF EXISTS {BACKFILL_UNIQUE_CATALOG_TABLE}")
    spark.sql(f"CREATE TABLE {BACKFILL_UNIQUE_CATALOG_TABLE} USING hudi LOCATION '{BACKFILL_UNIQUE_TABLE_PATH}'")

    create_index_options = [
        f"'vector.dimension' = '{BACKFILL_UNIQUE_DIMENSION}'",
        f"'vector.num_clusters' = '{BACKFILL_UNIQUE_NUM_CLUSTERS}'",
        f"'vector.metric' = '{BACKFILL_UNIQUE_METRIC}'",
        f"'vector.algorithm' = '{BACKFILL_UNIQUE_ALGORITHM}'",
        "'vector.rabitq.materialize.on.create' = 'true'",
    ]
    if VECTOR_QUANTIZER:
        create_index_options.append(f"'vector.quantizer' = '{VECTOR_QUANTIZER}'")

    create_index_sql = f"""
    CREATE INDEX {BACKFILL_UNIQUE_INDEX_NAME}
    ON {BACKFILL_UNIQUE_CATALOG_TABLE}
    USING VECTOR (embedding)
    OPTIONS (
      {', '.join(create_index_options)}
    )
    """

    print("BACKFILL_UNIQUE_RUN_ID =", BACKFILL_UNIQUE_RUN_ID)
    print("BACKFILL_UNIQUE_TABLE_NAME =", BACKFILL_UNIQUE_TABLE_NAME)
    print("BACKFILL_UNIQUE_TABLE_PATH =", BACKFILL_UNIQUE_TABLE_PATH)
    print("BACKFILL_UNIQUE_CATALOG_TABLE =", BACKFILL_UNIQUE_CATALOG_TABLE)
    print("Running CREATE INDEX with RaBitQ backfill enabled:")
    print(create_index_sql)
    spark.sql(create_index_sql)
    print("SHOW INDEXES output:")
    spark.sql(f"SHOW INDEXES FROM {BACKFILL_UNIQUE_CATALOG_TABLE}").show(truncate=False)
    print("Unique-path backfill table setup finished.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

In [ ]:
# Verify whether CREATE INDEX with backfill enabled actually wrote the hidden
# RaBitQ columns for the latest row version in the unique-path table.

from pyspark.sql import Window
from pyspark.sql.functions import col, desc, expr, length, row_number

if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    def logical_index_name(index_name):
        return index_name.removeprefix("vector_index_")

    def rabitq_binary_column(index_name):
        return f"_hudi_vec_{logical_index_name(index_name)}_binary_code"

    def rabitq_scalar_column(index_name):
        return f"_hudi_vec_{logical_index_name(index_name)}_scalar"

    spark = (
        SparkSession.builder.remote(SPARK_REMOTE)
        .config("spark.sql.parquet.enableVectorizedReader", PARQUET_VECTORIZED_READER)
        .config("hoodie.write.lock.provider", LOCK_PROVIDER)
        .getOrCreate()
    )
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")

    binary_col = rabitq_binary_column(BACKFILL_UNIQUE_INDEX_NAME)
    scalar_col = rabitq_scalar_column(BACKFILL_UNIQUE_INDEX_NAME)
    raw_all_df = (
        spark.read.format("parquet")
        .option("mergeSchema", "true")
        .load(os.environ.get("HOODIE_VECTOR_BACKFILL_UNIQUE_RAW_DATA_GLOB", f"{BACKFILL_UNIQUE_TABLE_PATH}/*.parquet"))
    )
    latest_window = Window.partitionBy("_hoodie_record_key").orderBy(desc("_hoodie_commit_time"), desc("_hoodie_commit_seqno"))
    raw_df = (
        raw_all_df
        .withColumn("_latest_rank", row_number().over(latest_window))
        .where(col("_latest_rank") == 1)
        .drop("_latest_rank")
        .cache()
    )
    row_count = raw_df.count()
    available_columns = sorted(raw_df.columns)
    quantizer_row = spark.sql(
        f"""
        SELECT
          VectorIndexMetadata.quantizerType AS quantizer_type,
          VectorIndexMetadata.quantizedCodeBytes AS quantized_code_bytes,
          VectorIndexMetadata.randomSeed AS random_seed,
          VectorIndexMetadata.assumeNormalized AS assume_normalized
        FROM hudi_metadata('{BACKFILL_UNIQUE_TABLE_PATH}')
        WHERE type = 8
          AND key = '__quantizer__'
        """
    ).collect()[0]

    print("BACKFILL_UNIQUE_TABLE_PATH =", BACKFILL_UNIQUE_TABLE_PATH)
    print("latest_row_count =", row_count)
    print("visible_columns =", available_columns)
    print("quantizer_row =", quantizer_row)

    assert row_count > 0, "No latest rows found in the unique-path backfill test table"
    assert binary_col in raw_df.columns, (
        f"Expected backfilled RaBitQ binary column {binary_col} in raw Parquet schema"
    )
    if not bool(quantizer_row.assume_normalized):
        assert scalar_col in raw_df.columns, (
            f"Expected backfilled RaBitQ scalar column {scalar_col} in raw Parquet schema"
        )

    stats_exprs = [
        expr("count(*)").alias("rows_total"),
        expr(f"sum(case when {binary_col} is not null then 1 else 0 end)").alias("rows_with_binary"),
        expr(f"sum(case when length({binary_col}) > 0 then 1 else 0 end)").alias("rows_with_nonempty_binary"),
    ]
    if scalar_col in raw_df.columns:
        stats_exprs.append(expr(f"sum(case when {scalar_col} is not null then 1 else 0 end)").alias("rows_with_scalar"))
    hidden_stats = raw_df.select(*stats_exprs).collect()[0]

    print("hidden_column_stats =", hidden_stats)
    print("Hidden-column samples from latest rows:")
    sample_columns = ["id", length(col(binary_col)).alias("binary_code_bytes")]
    if scalar_col in raw_df.columns:
        sample_columns.append(col(scalar_col).alias("scalar"))
    raw_df.select(*sample_columns).orderBy("id").show(truncate=False)

    assert hidden_stats.rows_with_binary == hidden_stats.rows_total, (
        f"Expected every latest row to be backfilled with {binary_col}; got {hidden_stats.rows_with_binary}/{hidden_stats.rows_total}"
    )
    assert hidden_stats.rows_with_nonempty_binary == hidden_stats.rows_total, (
        f"Expected every latest row to have a non-empty RaBitQ binary code; got {hidden_stats.rows_with_nonempty_binary}/{hidden_stats.rows_total}"
    )
    if scalar_col in raw_df.columns:
        assert hidden_stats.rows_with_scalar == hidden_stats.rows_total, (
            f"Expected every latest row to be backfilled with {scalar_col}; got {hidden_stats.rows_with_scalar}/{hidden_stats.rows_total}"
        )

    print("Unique-path backfill-on-create RaBitQ verification passed.")
    raw_df.unpersist()
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

In [ ]:
# Verify that the fresh post-index rows physically carry the hidden RaBitQ
# columns in the base Parquet files.

import math
import struct

from pyspark.sql.functions import col, expr, length, udf
from pyspark.sql.types import DoubleType

VECTOR_INDEX_NAME = os.environ.get("HOODIE_VECTOR_INDEX_NAME", "embedding_idx")
RABITQ_INSERT_LABEL = os.environ.get("HOODIE_VECTOR_RABITQ_INSERT_LABEL", "rabitq-fresh")
RAW_DATA_GLOB = os.environ.get("HOODIE_VECTOR_RAW_DATA_GLOB", f"{TABLE_PATH}/*.parquet")
QUERY_VECTOR_ENV = os.environ.get("HOODIE_VECTOR_QUERY_VECTOR")
VECTOR_ELEMENT_TYPE = os.environ.get("HOODIE_VECTOR_ELEMENT_TYPE", "FLOAT").upper()
RABITQ_APPROX_DISTANCE_FN = "hudi_rabitq_distance"
RABITQ_APPROX_DISTANCE_UDF_CLASS = "org.apache.hudi.sql.RaBitQApproxDistanceUDF"


def logical_index_name(index_name):
    return index_name.removeprefix("vector_index_")


def rabitq_binary_column(index_name):
    return f"_hudi_vec_{logical_index_name(index_name)}_binary_code"


def rabitq_scalar_column(index_name):
    return f"_hudi_vec_{logical_index_name(index_name)}_scalar"


def vector_dimension_from_bytes(raw_bytes: bytes, element_type: str) -> int:
    if raw_bytes is None:
        raise ValueError("Cannot infer vector dimension from null bytes")
    element_size = 4 if element_type == "FLOAT" else 8 if element_type == "DOUBLE" else 1
    if len(raw_bytes) % element_size != 0:
        raise ValueError(f"Byte length {len(raw_bytes)} is not divisible by element size {element_size}")
    return len(raw_bytes) // element_size


def parse_query_vector(dim: int):
    if QUERY_VECTOR_ENV:
        values = [float(value.strip()) for value in QUERY_VECTOR_ENV.split(",") if value.strip()]
        if len(values) != dim:
            raise ValueError(f"HOODIE_VECTOR_QUERY_VECTOR has {len(values)} values, expected {dim}")
        return values
    return [10.1] + [0.0] * (dim - 1)


def make_binary_l2_udf(query_vector, element_type: str):
    if element_type == "FLOAT":
        fmt_prefix = "<f"
        element_size = 4
    elif element_type == "DOUBLE":
        fmt_prefix = "<d"
        element_size = 8
    elif element_type == "INT8":
        fmt_prefix = "<b"
        element_size = 1
    else:
        raise ValueError(f"Unsupported vector element type: {element_type}")

    query_values = [float(value) for value in query_vector]

    def binary_l2(raw_bytes):
        if raw_bytes is None:
            return None
        dim = len(raw_bytes) // element_size
        values = [struct.unpack_from(fmt_prefix, raw_bytes, offset=i * element_size)[0] for i in range(dim)]
        return float(math.sqrt(sum((float(v) - query_values[i]) ** 2 for i, v in enumerate(values))))

    return udf(binary_l2, DoubleType())


def register_rabitq_distance_udf(spark_session):
    spark_session.udf.registerJavaFunction(
        RABITQ_APPROX_DISTANCE_FN, RABITQ_APPROX_DISTANCE_UDF_CLASS, "float"
    )
    return True


if "spark" not in globals():
    print("Spark session not initialized; run the Spark setup cells first.")
else:
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")

    rabitq_binary_col = rabitq_binary_column(VECTOR_INDEX_NAME)
    rabitq_scalar_col = rabitq_scalar_column(VECTOR_INDEX_NAME)
    quantizer_row = spark.sql(
        f"""
        SELECT
          VectorIndexMetadata.quantizerType AS quantizer_type,
          VectorIndexMetadata.quantizedCodeBytes AS quantized_code_bytes,
          VectorIndexMetadata.randomSeed AS random_seed,
          VectorIndexMetadata.assumeNormalized AS assume_normalized
        FROM hudi_metadata('{TABLE_PATH}')
        WHERE type = 8
          AND key = '__quantizer__'
        """
    ).collect()[0]

    raw_fresh_df = (
        spark.read.format("parquet")
        .option("mergeSchema", "true")
        .load(RAW_DATA_GLOB)
        .where(col("label") == RABITQ_INSERT_LABEL)
        .cache()
    )
    fresh_row_count = raw_fresh_df.count()
    available_columns = sorted(raw_fresh_df.columns)

    print("RAW_DATA_GLOB =", RAW_DATA_GLOB)
    print("fresh_label =", RABITQ_INSERT_LABEL)
    print("fresh_row_count =", fresh_row_count)
    print("visible_columns =", available_columns)
    print("quantizer_row =", quantizer_row)

    assert fresh_row_count > 0, "No fresh rows found in raw Parquet; run the insert cell first."
    assert rabitq_binary_col in raw_fresh_df.columns, (
        f"Expected hidden RaBitQ binary column {rabitq_binary_col} in raw Parquet schema"
    )
    if not bool(quantizer_row.assume_normalized):
        assert rabitq_scalar_col in raw_fresh_df.columns, (
            f"Expected hidden RaBitQ scalar column {rabitq_scalar_col} in raw Parquet schema"
        )

    stats_exprs = [
        expr("count(*)").alias("fresh_rows"),
        expr(f"sum(case when {rabitq_binary_col} is not null then 1 else 0 end)").alias("rows_with_binary"),
        expr(f"sum(case when length({rabitq_binary_col}) > 0 then 1 else 0 end)").alias("rows_with_nonempty_binary"),
    ]
    if rabitq_scalar_col in raw_fresh_df.columns:
        stats_exprs.append(expr(f"sum(case when {rabitq_scalar_col} is not null then 1 else 0 end)").alias("rows_with_scalar"))

    hidden_stats = raw_fresh_df.select(*stats_exprs).collect()[0]
    print("hidden_column_stats =", hidden_stats)

    sample_columns = ["id", length(col(rabitq_binary_col)).alias("binary_code_bytes")]
    if rabitq_scalar_col in raw_fresh_df.columns:
        sample_columns.append(col(rabitq_scalar_col).alias("scalar"))
    print("Hidden-column samples:")
    raw_fresh_df.select(*sample_columns).orderBy("id").show(truncate=False)

    assert hidden_stats.rows_with_binary == hidden_stats.fresh_rows, (
        f"Expected every fresh row to have {rabitq_binary_col}; got {hidden_stats.rows_with_binary}/{hidden_stats.fresh_rows}"
    )
    assert hidden_stats.rows_with_nonempty_binary == hidden_stats.fresh_rows, (
        f"Expected every fresh row to have a non-empty RaBitQ binary code; got {hidden_stats.rows_with_nonempty_binary}/{hidden_stats.fresh_rows}"
    )
    if rabitq_scalar_col in raw_fresh_df.columns:
        assert hidden_stats.rows_with_scalar == hidden_stats.fresh_rows, (
            f"Expected every fresh row to have {rabitq_scalar_col}; got {hidden_stats.rows_with_scalar}/{hidden_stats.fresh_rows}"
        )

    print("Raw-Parquet hidden-column verification passed.")
    raw_fresh_df.unpersist()

## 11. Functional RaBitQ ranking check

This section uses the hidden columns from the fresh rows to compute **RaBitQ approximate distance** and compares that ordering with **exact L2 distance** computed from the raw vector bytes.

That gives a focused functional answer:

- the RaBitQ code was generated
- the scoring UDF can consume it successfully
- the approximate ranking is sensible relative to exact distance on the same inserted slice

This deliberately validates **RaBitQ encoding + scoring** even if the current Hudi read path still does not project hidden columns for a full native query flow.

In [ ]:
# Score only the fresh rows using the hidden RaBitQ columns, and compare the
# resulting ordering with exact L2 on the original vector bytes.

VECTOR_INDEX_NAME = os.environ.get("HOODIE_VECTOR_INDEX_NAME", "embedding_idx")
RABITQ_INSERT_LABEL = os.environ.get("HOODIE_VECTOR_RABITQ_INSERT_LABEL", "rabitq-fresh")
RAW_DATA_GLOB = os.environ.get("HOODIE_VECTOR_RAW_DATA_GLOB", f"{TABLE_PATH}/*.parquet")
TOP_K = int(os.environ.get("HOODIE_VECTOR_QUERY_TOPK", str(VECTOR_QUERY_TOPK)))


if "spark" not in globals():
    print("Spark session not initialized; run the Spark setup cells first.")
else:
    spark.sql(f"SET spark.sql.parquet.enableVectorizedReader={PARQUET_VECTORIZED_READER}")
    spark.sql(f"SET hoodie.write.lock.provider={LOCK_PROVIDER}")

    register_rabitq_distance_udf(spark)

    quantizer_row = spark.sql(
        f"""
        SELECT
          VectorIndexMetadata.quantizerType AS quantizer_type,
          VectorIndexMetadata.randomSeed AS random_seed,
          VectorIndexMetadata.assumeNormalized AS assume_normalized
        FROM hudi_metadata('{TABLE_PATH}')
        WHERE type = 8
          AND key = '__quantizer__'
        """
    ).collect()[0]
    assert quantizer_row.quantizer_type == "IVF_RABITQ", (
        f"Expected IVF_RABITQ quantizer, got {quantizer_row.quantizer_type}"
    )

    binary_col = rabitq_binary_column(VECTOR_INDEX_NAME)
    scalar_col = rabitq_scalar_column(VECTOR_INDEX_NAME)
    raw_fresh_base_df = (
        spark.read.format("parquet")
        .option("mergeSchema", "true")
        .load(RAW_DATA_GLOB)
        .where(col("label") == RABITQ_INSERT_LABEL)
    )
    available_columns = set(raw_fresh_base_df.columns)
    selected_columns = ["id", "label", "embedding", binary_col]
    if scalar_col in available_columns:
        selected_columns.append(scalar_col)
    raw_fresh_df = raw_fresh_base_df.select(*selected_columns).cache()
    fresh_row_count = raw_fresh_df.count()
    assert fresh_row_count > 0, "No fresh rows found; run the insert cell first."

    sample_bytes = raw_fresh_df.select("embedding").limit(1).collect()[0].embedding
    inferred_dim = vector_dimension_from_bytes(sample_bytes, VECTOR_ELEMENT_TYPE)
    query_vector = parse_query_vector(inferred_dim)
    query_literal = "[" + ", ".join(f"{float(v):.6f}" for v in query_vector) + "]"
    binary_l2_udf = make_binary_l2_udf(query_vector, VECTOR_ELEMENT_TYPE)
    scalar_sql = scalar_col if scalar_col in raw_fresh_df.columns else "CAST(NULL AS FLOAT)"
    approx_distance_sql = (
        f"{RABITQ_APPROX_DISTANCE_FN}("
        f"{binary_col}, "
        f"{scalar_sql}, "
        f"'{query_literal}', "
        f"{len(query_vector)}, "
        f"{int(quantizer_row.random_seed)}L, "
        f"{str(bool(quantizer_row.assume_normalized)).lower()})"
    )

    approx_rows = (
        raw_fresh_df
        .selectExpr("id", "label", f"{approx_distance_sql} AS approx_distance")
        .orderBy("approx_distance", "id")
        .limit(min(TOP_K, fresh_row_count))
        .collect()
    )
    exact_rows = (
        raw_fresh_df
        .withColumn("exact_distance", binary_l2_udf("embedding"))
        .select("id", "label", "exact_distance")
        .orderBy("exact_distance", "id")
        .limit(min(TOP_K, fresh_row_count))
        .collect()
    )
    approx_all_non_null = (
        raw_fresh_df
        .selectExpr(f"{approx_distance_sql} AS approx_distance")
        .where("approx_distance IS NOT NULL")
        .count()
    )

    approx_topk_ids = [row.id for row in approx_rows]
    exact_topk_ids = [row.id for row in exact_rows]
    topk_overlap = [row_id for row_id in approx_topk_ids if row_id in set(exact_topk_ids)]

    print("query_vector =", query_vector)
    print("fresh_row_count =", fresh_row_count)
    print("approx_all_non_null =", approx_all_non_null)
    print("approx_topk_ids =", approx_topk_ids)
    print("exact_topk_ids =", exact_topk_ids)
    print("topk_overlap =", topk_overlap)
    print("\nApproximate top-k rows:")
    for rank, row in enumerate(approx_rows, start=1):
        print(f"{rank}. id={row.id}, label={row.label}, approx_distance={row.approx_distance:.6g}")
    print("\nExact top-k rows:")
    for rank, row in enumerate(exact_rows, start=1):
        print(f"{rank}. id={row.id}, label={row.label}, exact_distance={row.exact_distance:.6g}")

    assert approx_all_non_null == fresh_row_count, (
        f"Expected RaBitQ distance for every fresh row, got {approx_all_non_null}/{fresh_row_count}"
    )
    assert exact_topk_ids[0] in approx_topk_ids, (
        f"Expected exact nearest neighbor {exact_topk_ids[0]} to appear in approx top-k {approx_topk_ids}"
    )
    assert topk_overlap, "Expected non-empty overlap between approximate and exact top-k sets"
    print("Functional RaBitQ scoring verification passed.")

    raw_fresh_df.unpersist()

## MDT vector index structure (current layout) 

This section reflects the **current MDT-native key layout** used by the vector index path.

### 1) Key families and meaning

- `M|<gen>`: generation manifest (active generation metadata)
- `C|<gen>|<cluster>`: per-cluster metadata (including `shardCount`, `vectorCount`, and file-group mapping)
- `A|<recordKey>`: latest assignment for each record key (cluster/shard routing)
- `P|<gen>|<cluster>|<shard>|<recordKey>`: posting rows carrying compressed code payload (`binaryCode`, `scalar`) and map-back info (`fileGroupId`, `partitionPath`)
.

### 2) How data is arranged for fast lookup

- **Routing step**: read `M|` once to resolve current generation.
- **Coarse prune**: select top clusters and fetch `C|gen|cluster` rows.
- **Bounded scan**: expand cluster to shard prefixes and scan `P|gen|cluster|shard|*`.
- **Approx score**: compute RaBitQ distance from posting payload in MDT.
- **Map-back**: use record key (or record index) to fetch final base-table rows for exact rerank.

This keeps ANN candidate selection in MDT and avoids full base-table scans.

In [3]:
# MDT structure inspection for handoff
# Assumes `spark` and `TABLE_PATH` are already initialized in earlier cells.

from pyspark.sql import functions as F

if "spark" not in globals():
    raise RuntimeError("Spark session `spark` is not defined. Run the Spark init cells first.")

mdt = spark.sql(f"SELECT key, type, VectorIndexMetadata FROM hudi_metadata('{TABLE_PATH}')")

# Focus on vector-index metadata rows only
# In this notebook branch these rows are emitted under numeric type=8.
vector_mdt = mdt.where("type = 8").cache()

print("TABLE_PATH:", TABLE_PATH)
print("Vector MDT rows:", vector_mdt.count())
print()

# 1) Key-family distribution
family_df = (
    vector_mdt
    .select(
        F.when(F.col("key").startswith("M|"), F.lit("M| (generation manifest)"))
         .when(F.col("key").startswith("C|"), F.lit("C| (cluster metadata)"))
         .when(F.col("key").startswith("A|"), F.lit("A| (assignment)"))
         .when(F.col("key").startswith("P|"), F.lit("P| (posting)"))
         .when(F.col("key") == "__manifest__", F.lit("legacy __manifest__"))
         .when(F.col("key") == "__centroids__", F.lit("legacy __centroids__"))
         .when(F.col("key") == "__quantizer__", F.lit("legacy __quantizer__"))
         .when(F.col("key").startswith("__fg__/"), F.lit("legacy __fg__/"))
         .otherwise(F.lit("other"))
         .alias("key_family")
    )
    .groupBy("key_family")
    .count()
    .orderBy(F.desc("count"))
)
print("=== Key-family counts ===")
family_df.show(50, truncate=False)

# 2) Manifest rows (new + legacy)
print("=== Manifest rows ===")
vector_mdt.where(F.col("key").startswith("M|") | (F.col("key") == "__manifest__")) \
    .select(
        "key",
        F.col("VectorIndexMetadata.entryType").alias("entryType"),
        F.col("VectorIndexMetadata.generationId").alias("generationId"),
        F.col("VectorIndexMetadata.quantizerType").alias("quantizerType"),
        F.col("VectorIndexMetadata.quantizedCodeBytes").alias("quantizedCodeBytes")
    ) \
    .orderBy(F.col("generationId").desc_nulls_last()) \
    .show(20, truncate=False)

# 3) Cluster rows (C|...)
print("=== Cluster rows (C|...) sample ===")
cluster_df = vector_mdt.where(F.col("key").startswith("C|")) \
    .select(
        "key",
        F.col("VectorIndexMetadata.generationId").alias("generationId"),
        F.col("VectorIndexMetadata.clusterId").alias("clusterId"),
        F.col("VectorIndexMetadata.shardCount").alias("shardCount"),
        F.col("VectorIndexMetadata.vectorCount").alias("vectorCount"),
        F.col("VectorIndexMetadata.fileGroupIds").alias("fileGroupIds")
    )
cluster_df.orderBy("clusterId").show(20, truncate=False)

print("=== Cluster shard/vector summary ===")
cluster_df.select("clusterId", "shardCount", "vectorCount") \
    .orderBy(F.desc("vectorCount")) \
    .show(20, truncate=False)

# 4) Posting rows (P|...) sample
print("=== Posting rows (P|...) sample ===")
posting_df = vector_mdt.where(F.col("key").startswith("P|")) \
    .select(
        "key",
        F.col("VectorIndexMetadata.generationId").alias("generationId"),
        F.col("VectorIndexMetadata.clusterId").alias("clusterId"),
        F.col("VectorIndexMetadata.shardId").alias("shardId"),
        F.length(F.col("VectorIndexMetadata.binaryCode")).alias("binaryCodeBytes"),
        F.col("VectorIndexMetadata.scalar").alias("scalar"),
        F.col("VectorIndexMetadata.fileGroupId").alias("fileGroupId")
    )
posting_df.show(20, truncate=False)

print("=== Posting distribution by (clusterId, shardId) ===")
posting_df.groupBy("clusterId", "shardId") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(40, truncate=False)

# 5) Assignment rows (A|...) sample
print("=== Assignment rows (A|...) sample ===")
vector_mdt.where(F.col("key").startswith("A|")) \
    .select(
        "key",
        F.col("VectorIndexMetadata.generationId").alias("generationId"),
        F.col("VectorIndexMetadata.clusterId").alias("clusterId"),
        F.col("VectorIndexMetadata.shardId").alias("shardId"),
        F.col("VectorIndexMetadata.fileGroupId").alias("fileGroupId"),
        F.col("VectorIndexMetadata.isDeleted").alias("isDeleted")
    ) \
    .show(20, truncate=False)

print("Inspection complete. Use this output in the Hudi team walkthrough to show:")
print("- generation control (M|)")
print("- cluster + shard topology (C|)")
print("- bounded posting scan surface (P|)")
print("- record assignment map (A|)")


TABLE_PATH: gs://mystique_qa/hudi-vector/vector_connect_it_demo
Vector MDT rows: 0

=== Key-family counts ===
+----------+-----+
|key_family|count|
+----------+-----+
+----------+-----+

=== Manifest rows ===


{"ts": "2026-04-03 18:16:06.684", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[FIELD_NOT_FOUND] No such struct field `entryType` in `clusterId`, `centroidBytes`, `fileGroupId`, `partitionPath`, `fileGroupIds`, `vectorCount`, `lastUpdatedTs`, `quantizerType`, `quantizedCodeBytes`, `randomSeed`, `assumeNormalized`, `isDeleted`. SQLSTATE: 42704\n\nJVM stacktrace:\norg.apache.spark.sql.AnalysisException\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.noSuchStructFieldInGivenFieldsError(QueryCompilationErrors.scala:2567)\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.findField(complexTypeExtractors.scala:83)\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.apply(complexTypeExtractors.scala:56)\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.$anonfun$resolveCandidates$2(package.scala:399)\n\tat scala.collection.LinearSeqOps.foldLeft(LinearSeq.scala:183)\n\tat scala.collection.LinearSeqOps.foldLeft$(LinearSeq.sc

AnalysisException: [FIELD_NOT_FOUND] No such struct field `entryType` in `clusterId`, `centroidBytes`, `fileGroupId`, `partitionPath`, `fileGroupIds`, `vectorCount`, `lastUpdatedTs`, `quantizerType`, `quantizedCodeBytes`, `randomSeed`, `assumeNormalized`, `isDeleted`. SQLSTATE: 42704

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.noSuchStructFieldInGivenFieldsError(QueryCompilationErrors.scala:2567)
	at org.apache.spark.sql.catalyst.expressions.ExtractValue$.findField(complexTypeExtractors.scala:83)
	at org.apache.spark.sql.catalyst.expressions.ExtractValue$.apply(complexTypeExtractors.scala:56)
	at org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.$anonfun$resolveCandidates$2(package.scala:399)
	at scala.collection.LinearSeqOps.foldLeft(LinearSeq.scala:183)
	at scala.collection.LinearSeqOps.foldLeft$(LinearSeq.scala:179)
	at scala.collection.immutable.List.foldLeft(List.scala:79)
	at org.apache.spark.sql.cat